# Tier 5 Solutions — Set 2: Genomic Foundation Models

Reference solutions for the genomic foundation model assignment set.

In [ ]:
import numpy as np
from collections import Counter
import pandas as pd

## Problem 1 Solution

In [ ]:
def kmer_embedding(seq: str, vocab: list[str], k: int) -> np.ndarray:
    tokens = [seq[i:i + k] for i in range(len(seq) - k + 1)]
    counts = Counter(tokens)
    vec = np.array([counts[v] for v in vocab], dtype=float)
    return vec / (vec.sum() + 1e-9)

alphabet = ["A", "C", "G", "T"]
vocab2 = [a + b for a in alphabet for b in alphabet]
vec = kmer_embedding("ATATCC", vocab2, 2)
print("shape:", vec.shape, "sum:", float(vec.sum()))

## Problem 2 Solution

In [ ]:
def mutate(seq: str, pos: int, alt: str) -> str:
    return seq[:pos] + alt + seq[pos + 1:]

def toy_model(seq: str) -> np.ndarray:
    return np.array([seq.count("TATAAA"), seq.count("GATA")], dtype=float)

def ism_scores(seq: str, pos: int, model):
    ref = seq[pos]
    baseline = model(seq)
    out = {}
    for alt in "ACGT":
        if alt == ref:
            continue
        out[alt] = model(mutate(seq, pos, alt)) - baseline
    return out

test = "A" * 40 + "TATAAA" + "A" * 40
print(ism_scores(test, 42, toy_model))

## Problem 3 Solution

In [ ]:
def donor_score(window: str) -> float:
    center = window[len(window)//2 - 1:len(window)//2 + 1]
    return 0.8 if center == "GT" else 0.1

def acceptor_score(window: str) -> float:
    center = window[len(window)//2 - 1:len(window)//2 + 1]
    return 0.8 if center == "AG" else 0.1

def splice_deltas(seq: str, pos: int, alt: str, w: int = 9) -> dict:
    half = w // 2
    start = max(0, pos - half)
    end = min(len(seq), pos + half + 1)
    ref_window = seq[start:end]
    alt_seq = seq[:pos] + alt + seq[pos + 1:]
    alt_window = alt_seq[start:end]

    d_ref, d_alt = donor_score(ref_window), donor_score(alt_window)
    a_ref, a_alt = acceptor_score(ref_window), acceptor_score(alt_window)

    return {
        "DS_DG": max(0.0, d_alt - d_ref),
        "DS_DL": max(0.0, d_ref - d_alt),
        "DS_AG": max(0.0, a_alt - a_ref),
        "DS_AL": max(0.0, a_ref - a_alt),
    }

example = "CCTGACTGGTGAGTCTCAGGTTAC"
print(splice_deltas(example, 9, "A"))

## Problem 4 Solution

In [ ]:
def integrated_priority(max_ds: float, expr_delta: float, chrom_delta: float) -> float:
    return 0.6 * max_ds + 0.25 * abs(expr_delta) + 0.15 * abs(chrom_delta)

print(integrated_priority(0.7, -0.3, 0.1))

## Problem 5 Solution

In [ ]:
def choose_structure_model(has_complex: bool, has_ligand_or_nucleic: bool, need_open_rosetta_workflow: bool) -> str:
    if has_ligand_or_nucleic:
        return "AlphaFold3"
    if need_open_rosetta_workflow:
        return "RoseTTAFold"
    if has_complex:
        return "AlphaFold3"
    return "AlphaFold2"

print(choose_structure_model(False, False, False))
print(choose_structure_model(True, False, False))

## Problem 6 Solution

In [ ]:
df = pd.DataFrame([
    {"var": "v1", "coding": 1, "max_ds": 0.12, "expr_delta": 0.45, "missense_prob": 0.80, "rarity_score": 0.90, "mean_plddt": 84, "interface_pae": 6.0},
    {"var": "v2", "coding": 0, "max_ds": 0.91, "expr_delta": 0.15, "missense_prob": 0.05, "rarity_score": 0.70, "mean_plddt": 65, "interface_pae": 9.0},
    {"var": "v3", "coding": 1, "max_ds": 0.20, "expr_delta": 0.30, "missense_prob": 0.65, "rarity_score": 0.85, "mean_plddt": 72, "interface_pae": 14.0},
])

def rank_variants(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["structure_priority"] = np.where(
        out["coding"] == 0,
        0.0,
        0.45 * out["missense_prob"] +
        0.25 * out["expr_delta"].abs() +
        0.20 * out["rarity_score"] +
        0.10 * out["max_ds"]
    )

    out["confidence_factor"] = (
        0.6 * (out["mean_plddt"] / 100.0) +
        0.4 * (1.0 - np.clip(out["interface_pae"] / 30.0, 0, 1))
    )

    out["final_priority"] = out["structure_priority"] * out["confidence_factor"]
    return out.sort_values("final_priority", ascending=False)

rank_variants(df)